In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import xmlrpc.client
import datetime
import pandas as pd
import pickle
import numpy as np

In [2]:
class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", 8008),
                            requestHandler=RequestHandler,
                            logRequests=True)

stats_server = xmlrpc.client.ServerProxy("http://127.0.0.1:8018/RPC2")

In [3]:
def add_log(event_type, details=""):
    """Отправляет событие на сервер статистики."""
    try:
        log_message = event_type if not details else f"{event_type}:{details}"
        stats_server.add_log(log_message)
    except Exception as e:
        print(f"[WARNING] Stats server unavailable: {e}")

In [4]:
# Ping
def ping():
    add_log('ping')
    return True
server.register_function(ping, 'ping')

# Время сервера
def now():
    add_log('now')
    return datetime.datetime.now()
server.register_function(now, 'now')

# Отображение типа
def show_type(arg):
    add_log('show_type')
    return (str(arg), str(type(arg)), arg)
server.register_function(show_type, 'type')

# Сумма и степень
def test_sum(a, b):
    add_log('test_sum')
    return a + b
server.register_function(test_sum, 'sum')

def test_pow(a, b):
    add_log('test_pow')
    return a ** b
server.register_function(test_pow, 'pow')

<function __main__.test_pow(a, b)>

In [5]:
def damerau_levenshtein_distance(s1, s2):
    """Вычисляет расстояние Дамерау-Левенштейна между двумя строками."""
    s1 = s1.lower()
    s2 = s2.lower()
    
    if s1 == s2:
        return 0
    
    len_s1 = len(s1)
    len_s2 = len(s2)
    
    if len_s1 == 0:
        return len_s2
    if len_s2 == 0:
        return len_s1
    
    # Создание матрицы (len_s1 + 1) x (len_s2 + 1)
    # d[i][j] = расстояние между s1[:i] и s2[:j]
    d = [[0] * (len_s2 + 1) for _ in range(len_s1 + 1)]
    
    # Инициализация первой строки и столбца
    for i in range(len_s1 + 1):
        d[i][0] = i  # удаление i символов из s1
    for j in range(len_s2 + 1):
        d[0][j] = j  # вставка j символов в s1
    
    # Заполнение матрицы
    for i in range(1, len_s1 + 1):
        for j in range(1, len_s2 + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1

            # алгоритм Вагнера - Фишера
            d[i][j] = min(
                d[i - 1][j] + 1,      # удаление
                d[i][j - 1] + 1,      # вставка
                d[i - 1][j - 1] + cost  # замена
            )
            
            # Проверка транспозиции (если текущие символы разные)
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    
    return d[len_s1][len_s2]

In [6]:
def is_similar(s1, s2, max_distance=1):
    """Определяет, похожи ли две строки с заданным допуском."""
    distance = damerau_levenshtein_distance(s1, s2)
    return distance <= max_distance

In [7]:
def black_list_check_extended(surname, name, patronym, birth_date):
    """
    Проверка клиента в черном списке по ФИО и дате рождения.
    """
    add_log('black_list_check', f"{surname} {name} {patronym}")
    
    frame = pd.read_csv('bad_boys2.csv', header=0, sep=',', encoding='utf8')
    
    surname_lower = surname.lower()
    name_lower = name.lower()
    patronym_lower = patronym.lower()
    birth_date_lower = birth_date.lower()
    
    # Проверка полного совпадения
    exact_match = (
        (frame['Surname'].str.lower() == surname_lower) &
        (frame['Name'].str.lower() == name_lower) &
        (frame['Patronym'].str.lower() == patronym_lower) &
        (frame['Birth'].str.lower() == birth_date_lower)
    )
    
    if any(exact_match):
        fio = f"{surname} {name} {patronym}"
        return f"{fio} ({birth_date}): yes"
    
    # Проверка частичного совпадения
    similar_records = []
    
    for idx, row in frame.iterrows():
        # Извлечение данных из строки
        row_surname = str(row['Surname']).lower()
        row_name = str(row['Name']).lower()
        row_patronym = str(row['Patronym']).lower()
        row_birth = str(row['Birth']).lower()
        
        if row_birth != birth_date_lower:
            continue
        
        surname_similar = is_similar(surname_lower, row_surname, max_distance=1)
        name_similar = is_similar(name_lower, row_name, max_distance=1)
        patronym_similar = is_similar(patronym_lower, row_patronym, max_distance=1)
        
        if (surname_similar or surname_lower == row_surname) and \
           (name_similar or name_lower == row_name) and \
           (patronym_similar or patronym_lower == row_patronym):
            
            similar_records.append({
                'Surname': row['Surname'],
                'Name': row['Name'],
                'Patronym': row['Patronym'],
                'Birth': row['Birth']
            })
    
    # Формирование результата
    fio = f"{surname} {name} {patronym}"
    
    if similar_records:
        # Формирование списка похожих записей
        similar_list = []
        for rec in similar_records:
            similar_list.append(f"{rec['Surname']} {rec['Name']} {rec['Patronym']} ({rec['Birth']})")
        
        similar_str = "; ".join(similar_list)
        return f"{fio} ({birth_date}): similar [{similar_str}]"
    else:
        return f"{fio} ({birth_date}): no"

server.register_function(black_list_check_extended, 'black_list_check')

<function __main__.black_list_check_extended(surname, name, patronym, birth_date)>

In [8]:
def send_back_inversion(bin_data):
    """
    Инверсия цвета изображения.
    Поддерживает:
      - цветное RGB: shape (M, N, 3)
      - монохромное: shape (M, N) или (M, N, 1)
    """
    add_log('color_inversion')
    
    img_arr = pickle.loads(bin_data.data)
    shape = img_arr.shape
    
    # Определяем размерность
    if len(shape) == 2:
        # Монохромное (M, N)
        height, width = shape
        for i in range(height):
            for j in range(width):
                img_arr[i][j] = 255 - int(img_arr[i][j])
    elif len(shape) == 3:
        height, width, channels = shape
        for i in range(height):
            for j in range(width):
                for k in range(channels):
                    img_arr[i][j][k] = 255 - int(img_arr[i][j][k])
    else:
        raise ValueError(f"Неподдерживаемая размерность: {shape}")
    
    pimg = pickle.dumps(img_arr)
    return xmlrpc.client.Binary(pimg)

server.register_function(send_back_inversion, 'color_inversion')

<function __main__.send_back_inversion(bin_data)>

**Бинаризация** — это превращение изображения в черно-белое: если значение пикселя **≥ порога**, то *255 (белый)*, иначе *0 (черный)*

In [9]:
def send_back_binarization(bin_data, threshold):
    """
    Бинаризация изображения по заданному порогу (1-255).
    Поддерживает цветные (M, N, 3) и монохромные (M, N) / (M, N, 1) изображения.
    """
    add_log('binarization', f"threshold={threshold}")
    
    if not (1 <= threshold <= 255):
        raise ValueError("Порог должен быть в диапазоне 1-255")
    
    img_arr = pickle.loads(bin_data.data)
    shape = img_arr.shape
    
    if len(shape) == 2:
        # Монохромное (M, N)
        height, width = shape
        for i in range(height):
            for j in range(width):
                img_arr[i][j] = 255 if int(img_arr[i][j]) >= threshold else 0
    elif len(shape) == 3:
        height, width, channels = shape
        result = np.zeros((height, width), dtype=np.uint8)
        coef_height = 0.299
        coef_width = 0.587
        coef_channels = 0.114
        for i in range(height):
            for j in range(width):
                gray = int(coef_height * img_arr[i][j][0] + coef_width * img_arr[i][j][1] + coef_channels * img_arr[i][j][2])
                result[i][j] = 255 if gray >= threshold else 0
        return xmlrpc.client.Binary(pickle.dumps(result))
    else:
        raise ValueError(f"Неподдерживаемая размерность: {shape}")
    
    pimg = pickle.dumps(img_arr)
    return xmlrpc.client.Binary(pimg)

server.register_function(send_back_binarization, 'binarization')

<function __main__.send_back_binarization(bin_data, threshold)>

In [10]:
def send_back_binarization_cloud(bin_data, threshold):
    """Бинаризация изображения + подсчёт процента пикселей выше порога."""
    add_log('binarization_cloud', f"threshold={threshold}")
    
    if not (1 <= threshold <= 255):
        raise ValueError("Порог должен быть в диапазоне 1-255")
    
    img_arr = pickle.loads(bin_data.data)
    shape = img_arr.shape

    height = shape[0]
    width = shape[1]
    total_pixels = height * width
    above_count = 0
    
    if len(shape) == 2:
        # Монохромное (M, N)
        result = np.zeros((height, width), dtype=np.uint8)
        
        for i in range(height):
            for j in range(width):
                if int(img_arr[i][j]) >= threshold:
                    result[i][j] = 255
                    above_count += 1
                else:
                    result[i][j] = 0
    
    elif len(shape) == 3:
        # Цветное (M, N, 3)
        result = np.zeros((height, width), dtype=np.uint8)
        
        coef_height = 0.299
        coef_width = 0.587
        coef_channels = 0.114
        
        for i in range(height):
            for j in range(width):
                gray = int(coef_height * img_arr[i][j][0] + 
                          coef_width * img_arr[i][j][1] + 
                          coef_channels * img_arr[i][j][2])
                
                if gray >= threshold:
                    result[i][j] = 255
                    above_count += 1
                else:
                    result[i][j] = 0
    
    else:
        raise ValueError(f"Неподдерживаемая размерность: {shape}")
    
    percentage = (above_count / total_pixels) * 100
    pimg = pickle.dumps(result)
    return (xmlrpc.client.Binary(pimg), percentage)

server.register_function(send_back_binarization_cloud, 'binarization_cloud')

<function __main__.send_back_binarization_cloud(bin_data, threshold)>

**Разворот относительно вертикали** — это *зеркальное* отражение

In [11]:
def send_back_vertical_flip(bin_data):
    """
    Разворот изображения относительно вертикальной оси 
    (зеркальное отражение слева-направо).
    Поддерживает цветные (M, N, 3) и монохромные (M, N) / (M, N, 1) изображения.
    """
    add_log('vertical_flip')
    
    img_arr = pickle.loads(bin_data.data)
    shape = img_arr.shape
    
    # Создаем копию для результата, чтобы не портить исходные данные при обмене
    if len(shape) == 2:
        height, width = shape
        result = np.zeros_like(img_arr)
        for i in range(height):
            for j in range(width):
                result[i][j] = img_arr[i][width - 1 - j]
    elif len(shape) == 3:
        height, width, channels = shape
        result = np.zeros_like(img_arr)
        for i in range(height):
            for j in range(width):
                for k in range(channels):
                    result[i][j][k] = img_arr[i][width - 1 - j][k]
    else:
        raise ValueError(f"Неподдерживаемая размерность: {shape}")
    
    pimg = pickle.dumps(result)
    return xmlrpc.client.Binary(pimg)

server.register_function(send_back_vertical_flip, 'vertical_flip')

<function __main__.send_back_vertical_flip(bin_data)>

In [ ]:
print("Listening on port 8008...")
server.serve_forever()

Listening on port 8008...
